# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [10]:
import pickle
with open("data_set.pkl", "rb") as f:
    data_set = pickle.load(f)
with open("labels.pkl", "rb") as f:
    labels = pickle.load(f)
with open("test_data_set.pkl", "rb") as f:
    test_data_set = pickle.load(f)  
with open("test_labels.pkl", "rb") as f:
    test_labels = pickle.load(f)
with open("unique_labels.pkl", "rb") as f:
    unique_labels = pickle.load(f)

In [8]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

no stored variable or alias data_set
no stored variable or alias labels
no stored variable or alias test_data_set
no stored variable or alias test_labels
no stored variable or alias unique_labels


## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [27]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [39]:
from itertools import combinations

def build_classifiers():
    all_classifiers = {
        "Linear Regression": LinearRegression(),
        "Nearest Neighbors": KNeighborsClassifier(),
        "Linear SVM": SVC(kernel='linear'),
        "Decision Tree": DecisionTreeClassifier(),
        "Naive Bayes": GaussianNB(),
        "QDA": QuadraticDiscriminantAnalysis()
    }
    
    results = []
    
    for combo_names in combinations(all_classifiers.keys(), 3):
        classifiers = [all_classifiers[name] for name in combo_names]
        
        for clf in classifiers:
            clf.fit(data_set, labels)
            
        output = [clf.predict(data_set) for clf in classifiers]
        output = np.array(output).T
        
        stacked_classifier = SVC()
        stacked_classifier.fit(output, labels.ravel())
        
        test_output = [clf.predict(test_data_set) for clf in classifiers]
        test_output = np.array(test_output).T
        
        predicted = stacked_classifier.predict(test_output)
        
        predicted = np.array(predicted, dtype=test_labels.dtype).ravel()
        
        acc = accuracy_score(test_labels, predicted)
        results.append((acc, combo_names, classifiers))
        
    results.sort(key=lambda x: x[0], reverse=True)
    
    print("List of 10 best combinations")
    for i, (acc, names, _) in enumerate(results[:10], 1):
        print(f"{i}. Accuracy: {acc:.4f} | Models: {names}")
        

    return results[0][2]

In [40]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
        
    output = np.array(output).T 
    
    stacked_classifier = SVC() 
    stacked_classifier.fit(output, labels.ravel())
    
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
        
    test_set = np.array(test_set).T
    predicted = stacked_classifier.predict(test_set)
    
    return predicted

In [44]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)

List of 10 best combinations
1. Accuracy: 1.0000 | Models: ('Linear Regression', 'Nearest Neighbors', 'Decision Tree')
2. Accuracy: 1.0000 | Models: ('Nearest Neighbors', 'Decision Tree', 'Naive Bayes')
3. Accuracy: 0.9500 | Models: ('Linear Regression', 'Nearest Neighbors', 'QDA')
4. Accuracy: 0.9500 | Models: ('Linear Regression', 'Linear SVM', 'Decision Tree')
5. Accuracy: 0.9500 | Models: ('Linear Regression', 'Linear SVM', 'QDA')
6. Accuracy: 0.9500 | Models: ('Linear Regression', 'Naive Bayes', 'QDA')
7. Accuracy: 0.9500 | Models: ('Nearest Neighbors', 'Linear SVM', 'Naive Bayes')
8. Accuracy: 0.9500 | Models: ('Nearest Neighbors', 'Linear SVM', 'QDA')
9. Accuracy: 0.9500 | Models: ('Nearest Neighbors', 'Decision Tree', 'QDA')
10. Accuracy: 0.9500 | Models: ('Nearest Neighbors', 'Naive Bayes', 'QDA')


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [54]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [55]:
def set_new_weights(model):
    # fill the code here (two lines)
    indicator = (model.predict(test_set) != test_labels).astype(int)
    return (1 + indicator) / np.sum(1 + indicator)

Train the classifier with the code below:

In [56]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00130548 0.00130548 0.00130548 0.00130548 0.00065274 0.00130548
 0.00130548 0.00130548 0.00130548 0.00130548 0.00130548 0.00130548
 0.00130548 0.00130548 0.00130548 0.00065274 0.00130548 0.00130548
 0.00130548 0.00130548 0.00130548 0.00065274 0.00130548 0.00065274
 0.00065274 0.00130548 0.00065274 0.00065274 0.00065274 0.00130548
 0.00065274 0.00065274 0.00130548 0.00130548 0.00130548 0.00065274
 0.00130548 0.00065274 0.00130548 0.00130548 0.00130548 0.00130548
 0.00130548 0.00130548 0.00065274 0.00130548 0.00130548 0.00065274
 0.00130548 0.00065274 0.00130548 0.00065274 0.00130548 0.00130548
 0.00065274 0.00065274 0.00065274 0.00130548 0.00130548 0.00065274
 0.00065274 0.00130548 0.00065274 0.00065274 0.00130548 0.00130548
 0.00130548 0.00130548 0.00130548 0.00065274 0.00130548 0.00130548
 0.00065274 0.00065274 0.00130548 0.00065274 0.00130548 0.00065274
 0.00065274 0.00065274 0.00130548 0.00130548 0.00065274 0.00065274
 0.00065274 0.00130548 0.00130548 0.00130548 0.00065274 0.0013

Set the validation data set:

In [57]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [58]:
def get_prediction(x):
    all_predictions = np.array([clf.predict(x) for clf in classifiers])
    voted_predictions = []
    for i in range(all_predictions.shape[1]):
        voted_predictions.append(np.bincount(all_predictions[:, i]).argmax())
        
    return voted_predictions

Test it:

In [62]:
prediction = get_prediction(validate_x)[0]

print(prediction)

1
